# Base 0 (Zero-Shot) Inference & Tracking Orchestrator

This notebook performs sparse repository cloning, installs dependencies in editable mode, runs unit tests, and executes the Base 0 inference and tracking runner in an isolated process to prevent kernel restart prompts.

In [1]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = 'feat/19-base0-evaluation'

# --- Environment Detection: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Sparse clone repository from feature branch
if not REPO_PATH.exists():
    print(f"Cloning {REPO_NAME} (branch {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Updating existing {REPO_NAME} repository...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Directory navigation failed. Current path: {current_dir}")

# 3. Install package in editable mode
print("Installing package in editable mode with [cloud] dependencies...")
%pip install -q -e .[cloud]

/kaggle/working
Cloning ia_article (branch feat/19-base0-evaluation)...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 4.52 KiB | 4.52 MiB/s, done.
/kaggle/working/ia_article
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 30 (delta 0), reused 14 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 4.81 MiB | 3.94 MiB/s, done.
Updating files: 100% (35/35), done.
/kaggle/working/ia_article/experiments
Installing package in editable mode with [cloud] dependencies...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━

## 2. Run Unit Tests

Execute all co-located unit tests (IOManager, BaseInferencePipeline, and Base0Runner) to ensure system stability.

In [2]:
%cd /kaggle/working/ia_article/experiments
!pytest src/

/kaggle/working/ia_article/experiments
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working/ia_article/experiments
configfile: pyproject.toml
plugins: cov-7.1.0, anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collected 66 items                                                             

src/data_preparation/test_parser.py ..                                   [  3%]
src/evaluation/test_metric.py ........................                   [ 39%]
src/inference/runners/test_run_base_0.py ........                        [ 51%]
src/inference/test_base_inference.py ......                              [ 60%]
src/pseudo_labeling/test_pseudo_labeler.py .......s                      [ 72%]
src/utils/test_homography.py ......                                      [ 81%]
src/utils/test_io_manager.py ............                                [100%]

=============================== wa

## 3. Execute Base 0 Inference & Tracking Runner

Runs the `run_base_0` module in a separate process (`!python3 -m ...`). This performs health checks, dataset discovery, ByteTrack OBB tracking, Google Drive uploads, and automatic session shutdown.

In [3]:
!python3 -m src.inference.runners.run_base_0

Dataset directory: /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge
Output directory:  /kaggle/working/output_base0
[IOManager] token.json not found at /kaggle/working/token.json. Downloading automatically from Drive (ID: 1Fjg-AIrIQ77g1JRtapE6XDb_A6CP_4q3)...
[IOManager] ✅ token.json saved to /kaggle/working/token.json
[Base0Runner] DOTA -> MTC Class Mapping: {9: 6, 10: 0}
[Base0Runner] Selected Tracker: bytetrack.yaml

HEALTH CHECK
  ✅ Metadata:   /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/split_metadata.csv (1088 rows, 218 val clips)
  ✅ Images:     /kaggle/input/datasets/alvaroquispeunsa/mtc-challenge/train-001/train (sample: ['v_6kd3qt7m40_0008.jpg', 'v_d93kakshvf_0036.jpg', 'v_gqi2xjy8wt_0000.jpg', 'v_kq269jyo7m_0048.jpg', 'v_5kiqxe9rtu_0005.jpg'])
  ✅ Output:     /kaggle/working/output_base0
  ✅ Model:      yolo26m-obb.pt (15 classes)
  ✅ Vehicles:   2 vehicle classes mapped
  ✅ Drive:      Service initialized
  ✅ GPU:        Tesla T4 (14.6 GB)
  Health Check Stat